# 03 · Explaining the forecast with SHAP

TreeSHAP via LightGBM `pred_contrib`. With the anchored target, contributions explain the deviation from the previous day's mean price:

forecast = prev-day mean + base + Σ SHAP

In [1]:
import numpy as np
import plotly.io as pio
import polars as pl

from energy_forecast import explain, features, forecast, plots
from energy_forecast.data import load

pio.templates.default = "plotly_white"
raw = load()
df = features.build(raw)
cfg = forecast.DEFAULT
model = forecast.fit(forecast.train_window(df, forecast.TEST[0], cfg), cfg)
test = df.filter(pl.col("day").is_between(*forecast.TEST))
shap = explain.shap_values(model, test, cfg)

In [2]:
total = shap.select(pl.sum_horizontal(*cfg.features, "base", "anchor")).to_series().to_numpy()
np.abs(total - forecast.predict(model, test, cfg)).max()

np.float64(1.3926637620897964e-12)

## Global importance

In [3]:
imp = explain.importance(shap, cfg)
plots.shap_importance(imp)

## Dependence

In [4]:
for f in imp["feature"].head(4):
    plots.shap_dependence(test, shap, f).show()

## Single days: cheapest and most expensive day of the test year

In [5]:
daily = test.group_by("day").agg(pl.col("price").mean()).sort("price")
cheap, expensive = daily["day"][0], daily["day"][-1]
cheap, expensive

(datetime.date(2026, 4, 5), datetime.date(2026, 10, 29))

In [6]:
plots.shap_day(shap, list(cfg.features), cheap)

In [7]:
plots.shap_day(shap, list(cfg.features), expensive)

## Wind, solar and load

Most of the model works through residual load (load − wind − solar), so feature SHAP cannot say how much each physical driver contributes.
Instead, the three TSO forecasts are treated as players in an exact Shapley decomposition: each is either kept or replaced by its test-year mean, all derived features are rebuilt, and the model is evaluated on all 2³ combinations.

forecast = reference (all three at their mean) + wind + solar + load

Effects are relative to an average hour: solar lowers the price at noon in summer and raises it at night, when there is less sun than on average.

In [8]:
drv = explain.drivers(model, raw, forecast.TEST, cfg)
check = drv.select(pl.sum_horizontal("reference", *explain.DRIVERS) - pl.col("pred")).to_series()
check.abs().max()

5.684341886080802e-14

In [9]:
plots.driver_profile(drv)

In [10]:
drv.select(pl.col(*explain.DRIVERS).abs().mean())

Wind,Solar,Load
f64,f64,f64
18.210014,30.879249,17.053569


In [11]:
plots.driver_day(drv, cheap)

In [12]:
plots.driver_day(drv, expensive)